# 🏦 Mean Shift and DBSCAN


<a href="https://colab.research.google.com/github/rubenfonnegra/machine_learning/blob/master/Sem_07/meanshift_dbscan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> 
<a href="https://github.com/rubenfonnegra/machine_learning/blob/master/Sem_07/meanshift_dbscan.ipynb" target="_parent"><img src="https://img.shields.io/badge/%E2%80%8B-Open%20in%20Github-blue?logo=github" alt="Open In Github"/></a> 


This notebook applies **Mean Shift** and **DBSCAN** to an already-preprocessed numerical bank-customer dataset.

### Learning objectives
- Standardize numerical variables for distance-based clustering.
- Explain and estimate Mean Shift `bandwidth`.
- Compare bandwidths using centroid-distance and silhouette plots.
- Explain DBSCAN `eps` and `min_samples`.
- Use dimensionality-informed parameter heuristics.
- Use a k-distance elbow curve and silhouette analysis to tune DBSCAN.


### Documentation

- [```MeanShift```](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.MeanShift.html) 
- [```DBSCAN```](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html) 


---

> **📘 Machine Learning**  
> **Author:** Rubén D. Fonnegra, Ph.D. \
> **Institution:** Institución Universitaria Pascual Bravo  
> © 2026 · Educational use with attribution

### Imports

In [ ]:
#@markdown #### **🛠️⚙️📦 Install complementary dependencies**. 

from tqdm.auto import tqdm
import subprocess, time, sys

LIB = "MLTools-1.2-py3-none-any.whl"
URL = "https://drive.google.com/uc?id=18Y834Tvtj_-Px9yNmbAB4yV4L0gcIZ20"

commands = [
    ("📦 Downloading resources", ["gdown", URL, "-O", LIB], 35),
    ("🔧 Installing dependencies", [sys.executable, "-m", "pip", "install", "-q", LIB], 55),
    ("🧹 Finishing", ["rm", "-f", LIB], 10)
]

print("⚙️ Iniciando configuración del entorno...\n")

try:
    with tqdm(total=100, desc="Preparando", bar_format="{desc}: {bar} {n:.0f}%") as bar:
        for label, command, weight in commands:
            bar.set_description(label)
            subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            for _ in range(weight):
                time.sleep(0.01)
                bar.update(1)

    print("\n✅ Configuración completada correctamente.")

except subprocess.CalledProcessError as e:
    print("\n❌ Error durante la configuración")
    print(f"Exit code: {e.returncode}")

    if e.stdout:
        print("\n📤 STDOUT:")
        print(e.stdout)

    if e.stderr:
        print("\n🔍 STDERR:")
        print(e.stderr)

    print("\n❌ No fue posible configurar el entorno. Ejecute nuevamente la celda.")


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import MeanShift, DBSCAN, estimate_bandwidth
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

from MLTools import evaluate_meanshift_bandwidths, evaluate_dbscan_grid, plot_clustering_metric, generate_cluster_colors

### Load the dataset

In [ ]:
CSV_PATH = _ 
bank_df = pd.read_csv(CSV_PATH)

columns = [
    "customer_age", "annual_income", "account_balance",
    "monthly_transactions", "avg_transaction_value",
    "credit_utilization", "digital_logins_monthly",
    "products_held", "customer_tenure_years"
]

bank_df = bank_df[columns].copy()
print("Shape:", bank_df.shape)
bank_df.head()

### Inspect the data

In [ ]:
bank_df.describe().round(2)

In [ ]:
bank_df.isna().sum()

### Standardize the variables

Although all variables are numerical, their scales are very different. Mean Shift and DBSCAN use distances, so we standardize:

$$
z=\frac{x-\mu}{\sigma}
$$

In [ ]:
X = bank_df.to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=columns)
X_scaled_df.describe().round(3)

## Part I - Mean Shift

### Choosing a bandwidth

`bandwidth` controls the spatial scale at which Mean Shift searches for density modes.

- Smaller bandwidth → usually more clusters.
- Larger bandwidth → usually fewer clusters.
- Very large bandwidth → potentially one cluster.

There is **no universal optimal formula**. Bandwidth depends on feature scale, density, sample size, and dimensionality.

For standardized data, a simple classroom starting heuristic is to inspect roughly **5–10% of the largest standardized feature range**, then search a broader neighborhood around it.

Dimensionality matters because Euclidean distance accumulates differences across dimensions:

$$
d(\mathbf{x},\mathbf{y})=
\sqrt{\sum_{j=1}^{D}(x_j-y_j)^2}
$$

As \(D\) grows, typical distances generally increase and become more concentrated. Therefore, a bandwidth appropriate in 2D may be too small in 9D.

### Initial bandwidth

In [ ]:
feature_ranges = X_scaled_df.max() - X_scaled_df.min()
largest_range = feature_ranges.max()

bandwidth_5pct = 0.05 * largest_range
bandwidth_10pct = 0.10 * largest_range
INITIAL_BANDWIDTH = _ * largest_range

print("Dimensions:", X_scaled.shape[1])
print("Largest feature range:", round(largest_range, 3))
print("5%:", round(bandwidth_5pct, 3))
print("10%:", round(bandwidth_10pct, 3))
print("Initial bandwidth:", round(INITIAL_BANDWIDTH, 3))

### Initial Mean Shift model

In [ ]:
meanshift = MeanShift( bandwidth=INITIAL_BANDWIDTH )
meanshift_labels = meanshift.fit_predict(X_scaled)

print("Clusters:", len(np.unique(meanshift_labels)))
pd.Series(meanshift_labels).value_counts().sort_index()

### Visualization

In [ ]:
# custom_colors = generate_cluster_colors(meanshift_labels, {0: 'blue', 1: 'red', 2: 'green'}, default_color='lightgrey')

plt.figure(figsize=(9, 6))
plt.scatter(
    X_scaled_df.iloc[:, 0], X_scaled_df.iloc[:, 1],
    c=meanshift_labels,
    edgecolors="black", alpha=0.7, cmap='Paired'
)
plt.xlabel(columns[0])
plt.ylabel(columns[1])
plt.title(f"Mean Shift — bandwidth={INITIAL_BANDWIDTH:.3f}")
plt.show()

### Evaluate candidate bandwidths

In [ ]:
bandwidth_candidates = np.linspace(
    max(bandwidth_5pct * 0.5, 0.05), bandwidth_10pct * 2.5, 10
)

meanshift_analysis = evaluate_meanshift_bandwidths(
    X_train = X_scaled,
    bandwidths = bandwidth_candidates,
    metric = "euclidean"
)

meanshift_analysis

### Centroid-distance analysis

In [ ]:
_, axes = plt.subplots(1,3,figsize=(15,4))


plot_clustering_metric(
    meanshift_analysis,
    x="bandwidth",
    y="mean_nearest_centroid_distance",
    title="Mean Shift — Distance Analysis",
    x_label="Bandwidth",
    y_label="Mean nearest-centroid distance",
    ax= axes[0]
)

plot_clustering_metric(
    meanshift_analysis,
    x="bandwidth",
    y="silhouette",
    title="Mean Shift — Silhouette Analysis",
    x_label="Bandwidth",
    y_label="Silhouette score",
    ax= axes[1]
)

plot_clustering_metric(
    meanshift_analysis,
    x="bandwidth",
    y="n_clusters",
    title="Mean Shift — Number of Clusters",
    x_label="Bandwidth",
    y_label="Number of clusters",
    ax= axes[2]
)

### Select and retrain Mean Shift

In [ ]:
valid_ms = meanshift_analysis.dropna(subset=["silhouette"])
best_ms = valid_ms.loc[valid_ms["silhouette"].idxmax()]

optimal_bandwidth = float(best_ms["bandwidth"])

print("Selected bandwidth:", round(optimal_bandwidth, 3))
print("Clusters:", int(best_ms["n_clusters"]))
print("Silhouette:", round(best_ms["silhouette"], 3))

meanshift_final = MeanShift( bandwidth=optimal_bandwidth )
meanshift_final_labels = meanshift_final.fit_predict(X_scaled)

## Part II - DBSCAN

### Choosing `eps` and `min_samples`

#### `eps`
`eps` is a neighborhood radius. Because it is a distance, it depends strongly on scale and dimensionality.

The **5–10% largest-range heuristic** can provide an initial search region, but for DBSCAN the **k-distance elbow curve** is generally more informative.

#### `min_samples`
Dimensionality provides useful starting values. For \(D\) features, common heuristics include:

$$
\text{min\_samples}\geq D+1
$$

and

$$
\text{min\_samples}\approx 2D
$$

For sufficiently large datasets, values near **1–3% of the sample count** can also be explored. Percentage rules are exploratory rather than universal and may become excessively large for very large datasets.

### Initial parameters

In [ ]:
D = X_scaled.shape[1]
N = X_scaled.shape[0]

INITIAL_EPS = 0.075 * largest_range
INITIAL_MIN_SAMPLES = 2 * D

print("Samples:", N)
print("Dimensions:", D)
print("Initial eps:", round(INITIAL_EPS, 3))
print("D + 1:", D + 1)
print("2D:", 2 * D)
print("Initial min_samples:", INITIAL_MIN_SAMPLES)

### Initial DBSCAN model

In [ ]:
dbscan = DBSCAN(
    eps=INITIAL_EPS,
    min_samples=INITIAL_MIN_SAMPLES,
    metric="euclidean"
)

dbscan_labels = dbscan.fit_predict(X_scaled)
non_noise = dbscan_labels[dbscan_labels != -1]

print("Clusters:", len(np.unique(non_noise)))
print("Noise points:", np.sum(dbscan_labels == -1))
print("Noise fraction:", round(np.mean(dbscan_labels == -1), 3))

pd.Series(dbscan_labels).value_counts().sort_index()

### Initial DBSCAN visualization

In [ ]:

# color_code = {0: 'tab:blue', 1: 'tab:orange', 2: 'tab:green', 3:'tab:red', 
#               4: 'tab:gray', 5:'tab:cyan', 6:'tab:purple', 7:'tab:brown', 
#               8:'tab:pink', 9:'tab:olive'}
# dbscan_colors = generate_cluster_colors(dbscan_labels, color_code, default_color='lightgrey')

plt.figure(figsize=(9, 6))

plt.scatter(
    X_scaled[:, 0], X_scaled[:, 1],
    c=dbscan_labels,
    edgecolors="black", alpha=0.7, cmap='Paired'
)
plt.xlabel(columns[0])
plt.ylabel(columns[1])
plt.title(
    f"DBSCAN — eps={INITIAL_EPS:.3f}, "
    f"min_samples={INITIAL_MIN_SAMPLES}"
)
plt.show()

### Core points analysis

For each observation, we compute the distance to its \(k\)-th nearest neighbor using

$$
k=\text{min\_samples}
$$

After sorting those distances, a visible bend provides a useful candidate for `eps`.

In [ ]:
neighbors = NearestNeighbors(
    n_neighbors=INITIAL_MIN_SAMPLES,
    metric="euclidean"
)
neighbors.fit( _ )

distances, _ = neighbors.kneighbors( _ )
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(9, 5))
plt.plot(k_distances)
plt.axhline(
    INITIAL_EPS,
    linestyle="--",
    label=f"Initial eps={INITIAL_EPS:.3f}"
)
plt.xlabel("Observations sorted by distance")
plt.ylabel(
    f"Distance to {INITIAL_MIN_SAMPLES}-th nearest neighbor"
)
plt.title("DBSCAN — k-Distance Elbow Plot")
plt.legend()
plt.show()

### Candidate parameter grid

In [ ]:
eps_candidates = np.linspace(INITIAL_EPS - 0.1, 1.8, 12)

min_samples_candidates = sorted({
    D + 1,
    2 * D,
    max(D + 1, int(round(0.01 * N))),
    max(D + 1, int(round(0.02 * N))),
    max(D + 1, int(round(0.03 * N)))
})

print("eps:", np.round(eps_candidates, 3))
print("min_samples:", min_samples_candidates)

### Evaluate DBSCAN combinations

In [ ]:
dbscan_analysis = evaluate_dbscan_grid(
    X=X_scaled,
    eps_values=eps_candidates,
    min_samples_values=min_samples_candidates,
    metric="euclidean",
    ignore_noise_for_silhouette=True
)

dbscan_analysis

### Plot analysis

In [ ]:
_, axes = plt.subplots(1,3,figsize=(15,4))

plot_clustering_metric(
    dbscan_analysis,
    x="eps",
    y="silhouette",
    group="min_samples",
    title="DBSCAN — Silhouette Analysis",
    x_label="eps",
    y_label="Silhouette score",
    ax=axes[0]
)

plot_clustering_metric(
    dbscan_analysis,
    x="eps",
    y="noise_fraction",
    group="min_samples",
    title="DBSCAN — Noise Fraction",
    x_label="eps",
    y_label="Noise fraction",
    ax=axes[1]
)

plot_clustering_metric(
    dbscan_analysis,
    x="eps",
    y="n_clusters",
    group="min_samples",
    title="DBSCAN — Number of Clusters",
    x_label="eps",
    y_label="Number of clusters",
    ax=axes[2]
)

### Select a practical DBSCAN solution

In [ ]:
top_scores = _

valid_dbscan = dbscan_analysis[
    (dbscan_analysis["n_clusters"] >= 2)
    & dbscan_analysis["silhouette"].notna()
    & (dbscan_analysis["noise_fraction"] <= 0.40)
].copy()

if valid_dbscan.empty:
    print(
        "No candidate satisfied the filters. "
        "Inspect the plots and broaden the search."
    )
else:
    ordered_dbscan = valid_dbscan.sort_values(by="silhouette", ascending=False).drop_duplicates(subset="silhouette")
    top_dbscan = ( ordered_dbscan.head(top_scores).reset_index(drop=True) )

    top_dbscan.index += 1
    top_dbscan.index.name = "rank"

    print("Top 3 DBSCAN configurations:")
    display(top_dbscan[ [
                            "eps",
                            "min_samples",
                            "n_clusters",
                            "silhouette",
                            "noise_fraction"
                        ] ].round(6))

### Retrain DBSCAN

In [ ]:
optimal_eps = _ 
optimal_min_samples = _

if not valid_dbscan.empty:
    dbscan_final = DBSCAN(
        eps=optimal_eps,
        min_samples=optimal_min_samples,
        metric="euclidean"
    )

    dbscan_final_labels = dbscan_final.fit_predict(X_scaled)

    plt.figure(figsize=(9, 6))
    plt.scatter(
        X_scaled[:, 0], X_scaled[:, 1],
        c=dbscan_final_labels,
        edgecolors="black", alpha=0.7, cmap = 'Paired'
    )
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.title(
        f"Final DBSCAN — eps={optimal_eps:.3f}, "
        f"min_samples={optimal_min_samples}"
    )
    plt.show()

## Practice

1. Change the Mean Shift bandwidth search interval.
2. Compare the best silhouette bandwidth with neighboring values.
3. Test \(D+1\), \(2D\), and \(3D\) for DBSCAN.
4. Select `eps` visually from the k-distance elbow and compare it with the grid results.
5. Remove features and observe how the distance scale changes.

In [ ]:
# Your code here


---

## 📄 Attribution

This notebook was developed by **Rubén D. Fonnegra** as educational material for Machine Learning courses at **Institución Universitaria Pascual Bravo**.
You may use, share, and adapt this material for educational purposes, provided that appropriate credit is given to the original author.

**Suggested citation:**
> Fonnegra Tarazona, R. D. (2026). *Mean Shift and DBSCAN: Machine Learning Notebook*. Institución Universitaria Pascual Bravo.